# π0.5 YAM DiT LoRA fine-tuning

이 notebook은 기존 OpenPI의 normalization, test, training entry point만 호출한다.

- Base model: `pi05_base`
- Trainable parameter: action expert(DiT)의 LoRA parameter만
- Frozen parameter: VLM 전체와 action expert의 non-LoRA parameter
- Optimization: VLM prefix에 `stop_gradient`를 적용해 forward 값은 유지하고 frozen VLM backward graph를 차단
- LoRA: `LORA_RANK`, `LORA_ALPHA`, `LORA_RSLORA`를 attention과 FFN에 동일하게 적용


In [ ]:
from pathlib import Path
import os
import subprocess
from typing import Final


DATASET_REPO_ID: str = (
    'nengjanggo/yam_pick_up_the_white_ethernet_cable_and_plug_it_into_the_black_ethernet_port'
)

BATCH_SIZE: int = 8

PEAK_LR: float = 2.5e-5
DECAY_LR: float = 2.5e-6
WARMUP_STEPS: int = 1000
NUM_TRAIN_STEPS: int = 15000
DECAY_STEPS: int = NUM_TRAIN_STEPS

SAVE_INTERVAL: int = 1000
KEEP_PERIOD: int = 5000
OVERWRITE_CHECKPOINT: bool = True

LORA_RANK: int = 32
LORA_ALPHA: float = 32.0
LORA_RSLORA: bool = False

NUM_WORKERS: int = 1

NORM_MAX_FRAMES: int | None = None

RUN_TESTS: bool = False
RUN_NORM_STATS: bool = True
RUN_SMOKE_TRAINING: bool = False
RUN_FULL_TRAINING: bool = True

WANDB_ENABLED: bool = True
EXP_NAME: str = 'white_ethernet_cable_dit_lora'
RESUME: bool = False

XLA_MEMORY_FRACTION: str = '0.90'

LEROBOT_HOME: Path = Path.cwd().resolve() / 'lerobot-v3'
DATASET_DIR: Path = LEROBOT_HOME / DATASET_REPO_ID
CONFIG_NAME: Final[str] = 'pi05_yam_dit_lora'
ROOT_DIR: Final[Path] = Path.cwd().resolve()
OPENPI_DIR: Final[Path] = ROOT_DIR / 'third_party/yam-abc-reproduce/third_party/policy/openpi'


In [ ]:
# repository, dataset, 실행 mode를 학습 전에 검증한다.
assert (ROOT_DIR / 'yam.ipynb').is_file()
assert (OPENPI_DIR / 'scripts/train.py').is_file()
assert (DATASET_DIR / 'meta/info.json').is_file()
assert not (RESUME and OVERWRITE_CHECKPOINT)
assert not (RUN_SMOKE_TRAINING and RUN_FULL_TRAINING)

# OpenPI subprocess가 local LeRobot dataset과 GPU memory 설정을 사용하도록 구성한다.
training_env: dict[str, str] = os.environ.copy()
training_env['HF_LEROBOT_HOME'] = str(LEROBOT_HOME)
training_env['XLA_PYTHON_CLIENT_MEM_FRACTION'] = XLA_MEMORY_FRACTION


In [ ]:
if RUN_TESTS:
    # 새 YAM adapter, DiT LoRA filter, stop_gradient equivalence test만 실행한다.
    test_command: list[str] = [
        'uv',
        'run',
        'pytest',
        '-q',
        'src/openpi/policies/yam_policy_test.py',
        'src/openpi/training/yam_config_test.py',
        'src/openpi/models/model_test.py::test_pi05_stop_gradient_vlm_prefix_preserves_loss_and_action_expert_gradients',
    ]
    test_result: subprocess.CompletedProcess[str] = subprocess.run(
        test_command,
        cwd=OPENPI_DIR,
        env=training_env,
        check=True,
        text=True,
    )


In [ ]:
if RUN_NORM_STATS:
    # 전체 local dataset으로 π0.5 quantile normalization stats를 계산한다.
    norm_command: list[str] = [
        'uv',
        'run',
        'scripts/compute_norm_stats.py',
        '--config-name',
        CONFIG_NAME,
        '--repo-id',
        DATASET_REPO_ID,
    ]
    if NORM_MAX_FRAMES is not None:
        norm_command.extend(['--max-frames', str(NORM_MAX_FRAMES)])

    norm_result: subprocess.CompletedProcess[str] = subprocess.run(
        norm_command,
        cwd=OPENPI_DIR,
        env=training_env,
        check=True,
        text=True,
    )


In [ ]:
if RUN_SMOKE_TRAINING:
    # base checkpoint loading과 한 step update만 검증하고 별도 checkpoint에 저장한다.
    smoke_command: list[str] = [
        'uv',
        'run',
        'scripts/train.py',
        CONFIG_NAME,
        '--exp-name=smoke_dit_lora',
        f'--data.repo-id={DATASET_REPO_ID}',
        '--batch-size=1',
        f'--model.action-expert-lora-rank={LORA_RANK}',
        f'--model.action-expert-lora-alpha={LORA_ALPHA}',
        '--model.action-expert-lora-rslora' if LORA_RSLORA else '--model.no-action-expert-lora-rslora',
        f'--lr-schedule.peak-lr={PEAK_LR}',
        f'--lr-schedule.decay-lr={DECAY_LR}',
        f'--lr-schedule.warmup-steps={WARMUP_STEPS}',
        f'--lr-schedule.decay-steps={DECAY_STEPS}',
        '--num-train-steps=1',
        '--save-interval=1',
        '--num-workers=0',
        '--overwrite',
        '--no-wandb-enabled',
    ]
    smoke_result: subprocess.CompletedProcess[str] = subprocess.run(
        smoke_command,
        cwd=OPENPI_DIR,
        env=training_env,
        check=True,
        text=True,
    )


In [ ]:
if RUN_FULL_TRAINING:
    # 기존 OpenPI JAX training entry point에 notebook hyperparameter만 전달한다.
    train_command: list[str] = [
        'uv',
        'run',
        'scripts/train.py',
        CONFIG_NAME,
        f'--exp-name={EXP_NAME}',
        f'--data.repo-id={DATASET_REPO_ID}',
        f'--batch-size={BATCH_SIZE}',
        f'--model.action-expert-lora-rank={LORA_RANK}',
        f'--model.action-expert-lora-alpha={LORA_ALPHA}',
        '--model.action-expert-lora-rslora' if LORA_RSLORA else '--model.no-action-expert-lora-rslora',
        f'--lr-schedule.peak-lr={PEAK_LR}',
        f'--lr-schedule.decay-lr={DECAY_LR}',
        f'--lr-schedule.warmup-steps={WARMUP_STEPS}',
        f'--lr-schedule.decay-steps={DECAY_STEPS}',
        f'--num-train-steps={NUM_TRAIN_STEPS}',
        f'--save-interval={SAVE_INTERVAL}',
        f'--keep-period={KEEP_PERIOD}',
        f'--num-workers={NUM_WORKERS}',
    ]
    if RESUME:
        train_command.append('--resume')
    if OVERWRITE_CHECKPOINT:
        train_command.append('--overwrite')
    if not WANDB_ENABLED:
        train_command.append('--no-wandb-enabled')

    train_result: subprocess.CompletedProcess[str] = subprocess.run(
        train_command,
        cwd=OPENPI_DIR,
        env=training_env,
        check=True,
        text=True,
    )


In [ ]:
# 생성된 checkpoint step directory를 확인한다.
checkpoint_dir: Path = OPENPI_DIR / 'checkpoints' / CONFIG_NAME / EXP_NAME
checkpoint_steps: list[Path] = (
    sorted(path for path in checkpoint_dir.iterdir() if path.is_dir())
    if checkpoint_dir.is_dir()
    else []
)
checkpoint_steps
